In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                              accuracy_score, confusion_matrix, roc_curve)
from sklearn.dummy import DummyClassifier
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from pathlib import Path as _Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(
    style='whitegrid', palette='muted',
    rc={'figure.figsize': (12, 6), 'font.size': 12, 'axes.titlesize': 14}
)
COLORS = {'Absence': '#3498db', 'Presence': '#e74c3c'}
MODEL_COLORS = {
    'Baseline': '#95a5a6',
    'XGBoost':  '#e67e22',
    'LightGBM': '#9b59b6',
    'CatBoost': '#2ecc71',
}
RANDOM_STATE = 42
N_FOLDS = 5

# ── Load data (output of Notebook 2) ─────────────────────────────


train = pd.read_csv('train_processed.csv')
# test  = pd.read_csv('test_processed.csv')
print(f"Loaded processed data")
# ─────────────────────────────────────────────────────────────────────────────

feature_cols = [c for c in train.columns if c not in ['id', 'target']]
X      = train[feature_cols]
y      = train['target']
# X_test = test[feature_cols]

print(f"Features: {len(feature_cols)} | Data: {X.shape}")
print(f"Target balance: {y.mean():.3f} Presence rate")

Loaded processed data
Features: 18 | Data: (630000, 18)
Target balance: 0.448 Presence rate


In [8]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def cv_score(model, X, y, cv):
    """Return mean AUC, F1, Recall, Precision, Accuracy across CV folds."""
    results = cross_validate(model, X, y, cv=cv,
                             scoring=['roc_auc', 'f1', 'recall', 'precision', 'accuracy'],
                             n_jobs=-1)
    return {k.replace('test_', ''): round(v.mean(), 4)
            for k, v in results.items() if k.startswith('test_')}

In [9]:
dummy    = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
baseline = cv_score(dummy, X, y, skf)
print("Baseline (majority class predictor):")
for k, v in baseline.items():
    print(f"  {k}: {v:.4f}")

Baseline (majority class predictor):
  roc_auc: 0.5000
  f1: 0.0000
  recall: 0.0000
  precision: 0.0000
  accuracy: 0.5517


In [10]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
)
xgb_scores = cv_score(xgb_model, X, y, skf)
print("XGBoost CV:", xgb_scores)

XGBoost CV: {'roc_auc': 0.9552, 'f1': 0.8747, 'recall': 0.867, 'precision': 0.8826, 'accuracy': 0.8886}


In [11]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
)
lgb_scores = cv_score(lgb_model, X, y, skf)
print("LightGBM CV:", lgb_scores)

LightGBM CV: {'roc_auc': 0.955, 'f1': 0.8744, 'recall': 0.8675, 'precision': 0.8815, 'accuracy': 0.8883}


In [12]:
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
import catboost as cb
import pandas as pd

cat_features_idx = X.select_dtypes(include=['object', 'category']).columns.tolist()

class CatBoostWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, iterations=200, learning_rate=0.1, depth=6,
                 cat_features=None, random_state=42, verbose=0,
                 border_count=32, one_hot_max_size=10):
        self.iterations = iterations
        self.learning_rate = learning_rate
        self.depth = depth
        self.cat_features = cat_features
        self.random_state = random_state
        self.verbose = verbose
        self.border_count = border_count
        self.one_hot_max_size = one_hot_max_size

    def fit(self, X, y):
        try:
            device = 'GPU' if cb.get_gpu_device_count() > 0 else 'CPU'
        except:
            device = 'CPU'
            
        self.model_ = cb.CatBoostClassifier(
            iterations=self.iterations,
            learning_rate=self.learning_rate,
            depth=self.depth,
            cat_features=self.cat_features,
            random_state=self.random_state,
            verbose=self.verbose,
            task_type=device,
            border_count=self.border_count,
            one_hot_max_size=self.one_hot_max_size,
            thread_count=-1
        )
        self.model_.fit(X, y)
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

cb_model_wrapper = CatBoostWrapper(
    iterations=200, 
    learning_rate=0.07, 
    depth=6,
    cat_features=cat_features_idx,
    random_state=RANDOM_STATE, 
    verbose=0
)

# Bây giờ cv_score sẽ hoạt động bình thường vì Wrapper tuân thủ chuẩn sklearn
cb_scores = cv_score(cb_model_wrapper, X, y, skf)
print("CatBoost CV:", cb_scores)

CatBoost CV: {'roc_auc': 0.9543, 'f1': 0.8732, 'recall': 0.8649, 'precision': 0.8817, 'accuracy': 0.8874}


In [14]:
# modeling.py
import joblib
import os
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.base import BaseEstimator, ClassifierMixin

# --- 1. HÀM TẠO FEATURE (Giống hệt những gì bạn đã phân tích) ---
def create_features(df):
    df = df.copy()
    df['age_risk'] = pd.cut(df['Age'], bins=[0, 40, 55, 65, 120], labels=[0, 1, 2, 3]).astype(int)
    df['bp_chol_interaction'] = df['BP'] * df['Cholesterol'] / 10000
    df['hr_reserve_ratio'] = df['Max HR'] / (220 - df['Age']).replace(0, 1)
    df['stress_score'] = df['ST depression'] / (df['hr_reserve_ratio'] + 1e-6)
    df['silent_ischemia_flag'] = ((df['Chest pain type'] == 4) & (df['Exercise angina'] == 1)).astype(int)
    return df

# --- 2. CLASS WRAPPER (Giữ nguyên) ---
class CatBoostWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, iterations=200, learning_rate=0.1, depth=6,
                 cat_features=None, random_state=42, verbose=0,
                 border_count=32, one_hot_max_size=10):
        self.iterations = iterations
        self.learning_rate = learning_rate
        self.depth = depth
        self.cat_features = cat_features
        self.random_state = random_state
        self.verbose = verbose
        self.border_count = border_count
        self.one_hot_max_size = one_hot_max_size

    def fit(self, X, y):
        try:
            device = 'GPU' if cb.get_gpu_device_count() > 0 else 'CPU'
        except:
            device = 'CPU'
            
        self.model_ = cb.CatBoostClassifier(
            iterations=self.iterations,
            learning_rate=self.learning_rate,
            depth=self.depth,
            cat_features=self.cat_features,
            random_state=self.random_state,
            verbose=self.verbose,
            task_type=device,
            border_count=self.border_count,
            one_hot_max_size=self.one_hot_max_size,
            thread_count=-1
        )
        self.model_.fit(X, y)
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

# --- 3. DỮ LIỆU & HUẤN LUYỆN ---
# Giả sử X, y, RANDOM_STATE đã được định nghĩa từ phần tiền xử lý trước đó của bạn
# QUAN TRỌNG: Áp dụng create_features cho X trước khi train
print("Đang tạo feature engineering...")
X_processed = create_features(X) 

# Xác định categorical features trên tập dữ liệu ĐÃ XỬ LÝ
cat_features_idx = X_processed.select_dtypes(include=['object', 'category']).columns.tolist()

MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("🚀 Bắt đầu huấn luyện và lưu mô hình...")

# 1. XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1, use_label_encoder=False
)
xgb_model.fit(X_processed, y)
joblib.dump(xgb_model, os.path.join(MODEL_DIR, "xgb_model.pkl"))
print("✅ Đã lưu XGBoost")

# 2. LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
)
lgb_model.fit(X_processed, y)
joblib.dump(lgb_model, os.path.join(MODEL_DIR, "lgb_model.pkl"))
print("✅ Đã lưu LightGBM")

# 3. CatBoost
cb_model_wrapper = CatBoostWrapper(
    iterations=200, learning_rate=0.07, depth=6,
    cat_features=cat_features_idx,
    random_state=RANDOM_STATE, verbose=0
)
cb_model_wrapper.fit(X_processed, y)
joblib.dump(cb_model_wrapper, os.path.join(MODEL_DIR, "cb_model.pkl"))
print("✅ Đã lưu CatBoost")

# 4. Lưu Metadata (Lưu tên cột SAU KHI đã tạo feature)
metadata = {
    "feature_names": X_processed.columns.tolist(), # List 18 cột
    "cat_features_idx": cat_features_idx,
    "target_classes": np.unique(y).tolist()
}
joblib.dump(metadata, os.path.join(MODEL_DIR, "metadata.pkl"))
print("✅ Đã lưu Metadata")
print(f"\n📁 Hoàn tất! File lưu tại: {MODEL_DIR}")

Đang tạo feature engineering...
🚀 Bắt đầu huấn luyện và lưu mô hình...
✅ Đã lưu XGBoost
✅ Đã lưu LightGBM
✅ Đã lưu CatBoost
✅ Đã lưu Metadata

📁 Hoàn tất! File lưu tại: saved_models
